# Week 4: Create Joint General and Synthetic Training Data

This notebook creates a 200-question general dataset and a reduced synthetic dataset containing **700 total training examples** from the original 500 facts. It does not create 700 new facts.

The synthetic set contains 500 original training prompts plus 200 additional training paraphrases. Final testing uses 800 different held-out paraphrases.

## 1. Mount Google Drive and Set Paths

In [ ]:
import csv
import hashlib
import json
import random
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
THESIS_DIR = Path('/content/drive/MyDrive/Thesis')
SOURCE_GENERAL_DIR = (
    THESIS_DIR / 'Week 3' / 'data' / 'always_on_preservation_v2'
)
SOURCE_SYNTHETIC_DIR = (
    THESIS_DIR / 'Week 2' / 'data' / 'synthetic_facts_v1'
)
GENERAL_DIR = (
    THESIS_DIR / 'Week 4 - Joint Training Experiments' / 'data' / 'general_knowledge_200_v1'
)
SYNTHETIC_DIR = (
    THESIS_DIR / 'Week 4 - Joint Training Experiments' / 'data' / 'synthetic_examples_700_v2'
)
SEED = 42

required = [
    SOURCE_GENERAL_DIR / 'general_replay_v2.jsonl',
    SOURCE_GENERAL_DIR / 'general_validation_v2.jsonl',
    SOURCE_SYNTHETIC_DIR / 'train_all.jsonl',
    SOURCE_SYNTHETIC_DIR / 'eval_all.jsonl',
]
for path in required:
    assert path.exists(), f'Missing prerequisite file: {path}'

GENERAL_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
print('General output:', GENERAL_DIR)
print('Synthetic output:', SYNTHETIC_DIR)

## 2. Generate and Save Both Datasets

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8', newline='\n') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')

def write_csv(path, rows):
    clean = [
        {key: value for key, value in row.items() if key != 'messages'}
        for row in rows
    ]
    with Path(path).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(clean[0]))
        writer.writeheader()
        writer.writerows(clean)

# The fixed 200-question general dataset is already included in the organized
# Drive upload. Verify it rather than regenerating different questions.
general_manifest = json.loads(
    (GENERAL_DIR / 'manifest.json').read_text(encoding='utf-8')
)
for details in general_manifest['sets'].values():
    path = GENERAL_DIR / details['jsonl_file']
    assert path.exists(), f'Missing general dataset file: {path}'
    assert sha256(path) == details['jsonl_sha256']

base_train = read_jsonl(SOURCE_SYNTHETIC_DIR / 'train_all.jsonl')
all_eval = read_jsonl(SOURCE_SYNTHETIC_DIR / 'eval_all.jsonl')

alternate_by_fact = {}
for row in all_eval:
    prompt_index = int(row['example_id'].rsplit('_', 1)[1])
    if prompt_index == 0:
        continue
    key = (row['entity_id'], row['fact_type'])
    alternate_by_fact.setdefault(key, []).append(row)

rng = random.Random(SEED)
selected_keys = set()
for fact_type in sorted({row['fact_type'] for row in base_train}):
    for split, count in [('forget', 8), ('retain', 32)]:
        keys = sorted(
            (row['entity_id'], row['fact_type'])
            for row in base_train
            if row['fact_type'] == fact_type and row['split'] == split
        )
        rng.shuffle(keys)
        selected_keys.update(keys[:count])

extra_train = []
heldout_eval = []
for key, rows in sorted(alternate_by_fact.items()):
    rows = sorted(rows, key=lambda row: row['example_id'])
    if key in selected_keys:
        extra = dict(rows[0])
        extra['example_id'] = extra['example_id'].replace(
            'eval_', 'train_extra_', 1
        )
        extra_train.append(extra)
        heldout_eval.append(rows[1])
    else:
        heldout_eval.extend(rows)

synthetic_train = sorted(
    base_train + extra_train, key=lambda row: row['example_id']
)
heldout_eval = sorted(heldout_eval, key=lambda row: row['example_id'])
eval_forget = [row for row in heldout_eval if row['split'] == 'forget']
eval_retain = [row for row in heldout_eval if row['split'] == 'retain']

assert len(synthetic_train) == 700
assert len({(row['entity_id'], row['fact_type']) for row in synthetic_train}) == 500
assert len(heldout_eval) == 800
assert len(eval_forget) == 160
assert len(eval_retain) == 640
assert {row['prompt'] for row in synthetic_train}.isdisjoint(
    {row['prompt'] for row in heldout_eval}
)

sets = {
    'train_all': synthetic_train,
    'eval_all': heldout_eval,
    'eval_forget': eval_forget,
    'eval_retain': eval_retain,
}
manifest = {
    'dataset_name': 'synthetic_examples_700_v2',
    'seed': SEED,
    'source_dataset': 'synthetic_facts_v1',
    'num_identities': 100,
    'num_unique_facts': 500,
    'num_train_examples': 700,
    'num_extra_training_paraphrases': 200,
    'num_heldout_eval_examples': 800,
    'prompt_overlap_between_train_and_eval': 0,
    'sets': {},
}
for name, rows in sets.items():
    jsonl_path = SYNTHETIC_DIR / f'{name}.jsonl'
    csv_path = SYNTHETIC_DIR / f'{name}.csv'
    write_jsonl(jsonl_path, rows)
    write_csv(csv_path, rows)
    manifest['sets'][name] = {
        'num_records': len(rows),
        'jsonl_file': jsonl_path.name,
        'csv_file': csv_path.name,
        'jsonl_sha256': sha256(jsonl_path),
    }
(SYNTHETIC_DIR / 'manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(manifest, indent=2))

## 3. Verify Counts and Splits

In [ ]:
general_counts = {
    split: len(read_jsonl(GENERAL_DIR / f'general_{split}.jsonl'))
    for split in ['train', 'validation', 'test']
}
synthetic_counts = {
    name: len(read_jsonl(SYNTHETIC_DIR / f'{name}.jsonl'))
    for name in ['train_all', 'eval_all', 'eval_forget', 'eval_retain']
}

train_prompts = {
    row['prompt'] for row in read_jsonl(SYNTHETIC_DIR / 'train_all.jsonl')
}
eval_prompts = {
    row['prompt'] for row in read_jsonl(SYNTHETIC_DIR / 'eval_all.jsonl')
}
assert train_prompts.isdisjoint(eval_prompts)

print('General counts:', general_counts)
print('Synthetic counts:', synthetic_counts)
print('Synthetic train/eval prompt overlap:', len(train_prompts & eval_prompts))
print('Data creation complete.')

## Saved Files

General data: `MyDrive/Thesis/Week 4 - Joint Training Experiments/data/general_knowledge_200_v1`

Reduced synthetic data: `MyDrive/Thesis/Week 4 - Joint Training Experiments/data/synthetic_examples_700_v2`

The reduced synthetic set has 700 training examples and 800 held-out evaluation paraphrases.